# Heme-binding pilot: build the 80-protein master dataset

This notebook organises the **30 reference proteins**, **20 AI-designed heme binders**, **10 natural heme binders** and **20 negative controls**. It downloads UniProt sequences for 60 proteins and includes the 20 designed sequences directly.

The resulting CSV uses simple **positive/negative** labels. The study limitations will be discussed in the written research report, not in the dataset labels.

Run the cells in order. No BLASTp or ESM-2 evaluation is performed here.


In [ ]:
# Step 1: Load our 80 selected protein records and existing AI-designed sequences.
# The entries below contain the protein identities, groups and sequence sources.
import time
import requests
import pandas as pd
from collections import Counter
from IPython.display import display

# Assemble all protein records without downloading anything yet.
records = [{'record_id': 'P02144',
  'protein_name': 'Human myoglobin',
  'uniprot_accession': 'P02144',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Globin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P02144',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P68871',
  'protein_name': 'Human hemoglobin beta',
  'uniprot_accession': 'P68871',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Globin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P68871',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q9NPG2',
  'protein_name': 'Human neuroglobin',
  'uniprot_accession': 'Q9NPG2',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Globin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q9NPG2',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P02236',
  'protein_name': 'Soybean leghemoglobin',
  'uniprot_accession': 'P02236',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Globin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P02236',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P04252',
  'protein_name': 'Vitreoscilla hemoglobin',
  'uniprot_accession': 'P04252',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Globin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P04252',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P08684',
  'protein_name': 'Human CYP3A4',
  'uniprot_accession': 'P08684',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Cytochrome P450',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P08684',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P00183',
  'protein_name': 'Bacterial P450cam',
  'uniprot_accession': 'P00183',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Cytochrome P450',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P00183',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P10614',
  'protein_name': 'Yeast CYP51',
  'uniprot_accession': 'P10614',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Cytochrome P450',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P10614',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P92994',
  'protein_name': 'Plant CYP73A5',
  'uniprot_accession': 'P92994',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Cytochrome P450',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P92994',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q00441',
  'protein_name': 'Bacterial CYP107A1',
  'uniprot_accession': 'Q00441',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Cytochrome P450',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q00441',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P00433',
  'protein_name': 'Horseradish peroxidase',
  'uniprot_accession': 'P00433',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Peroxidase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P00433',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P00431',
  'protein_name': 'Yeast cytochrome c peroxidase',
  'uniprot_accession': 'P00431',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Peroxidase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P00431',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P05164',
  'protein_name': 'Human myeloperoxidase',
  'uniprot_accession': 'P05164',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Peroxidase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P05164',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P06181',
  'protein_name': 'Fungal lignin peroxidase',
  'uniprot_accession': 'P06181',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Peroxidase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P06181',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P13029',
  'protein_name': 'E. coli KatG',
  'uniprot_accession': 'P13029',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Peroxidase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P13029',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P99999',
  'protein_name': 'Human cytochrome c',
  'uniprot_accession': 'P99999',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Cytochrome',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P99999',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P00167',
  'protein_name': 'Human cytochrome b5',
  'uniprot_accession': 'P00167',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Cytochrome',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P00167',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P0ABE7',
  'protein_name': 'E. coli cytochrome b562',
  'uniprot_accession': 'P0ABE7',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Cytochrome',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P0ABE7',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P00156',
  'protein_name': 'Human mitochondrial cytochrome b',
  'uniprot_accession': 'P00156',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Cytochrome',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P00156',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P30519',
  'protein_name': 'Human HO-2',
  'uniprot_accession': 'P30519',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Heme oxygenase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P30519',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q54AI1',
  'protein_name': 'Bacterial HmuO',
  'uniprot_accession': 'Q54AI1',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Heme oxygenase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q54AI1',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q2FZE2',
  'protein_name': 'Staphylococcus IsdG',
  'uniprot_accession': 'Q2FZE2',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Heme oxygenase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q2FZE2',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q8X5N8',
  'protein_name': 'E. coli ChuS',
  'uniprot_accession': 'Q8X5N8',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Heme oxygenase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q8X5N8',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P04040',
  'protein_name': 'Human catalase',
  'uniprot_accession': 'P04040',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Catalase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P04040',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P21179',
  'protein_name': 'E. coli catalase HPII',
  'uniprot_accession': 'P21179',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Catalase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P21179',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q59635',
  'protein_name': 'Pseudomonas catalase KatB',
  'uniprot_accession': 'Q59635',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Catalase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q59635',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P02790',
  'protein_name': 'Human hemopexin',
  'uniprot_accession': 'P02790',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Other heme binder',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P02790',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q54450',
  'protein_name': 'Serratia HasA',
  'uniprot_accession': 'Q54450',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Other heme binder',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q54450',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'P29474',
  'protein_name': 'Human NOS3',
  'uniprot_accession': 'P29474',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Other heme binder',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P29474',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q02153',
  'protein_name': 'Human guanylate cyclase beta-1',
  'uniprot_accession': 'Q02153',
  'dataset_group': 'REFERENCE',
  'reference_family_or_control_type': 'Other heme binder',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q02153',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'REFERENCE',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb02',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb02',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'GLTEEEQKLVWEIFERFEEDLEGFGLDVLIRAFTEHPETLKKFPRFADLKSEAELRASPRLREHGVTVLKALIKIFKKGEDFAEEVKPLAESHSKVHKIPVSDLEVIAAAILATAKERFPEFFNEKAQAALTKALQQFIDAIAAEYKKL',
  'sequence_length': 149,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb03',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb03',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'GLTAEEQKLVRDIWAEVEKDREGFGLEVLLLTFTEHPETLKKFPRFAHLKSAEELRASPEAKAHGVTVLDALSKILKKGSNFEEEIKPLAESHYKEHKIPIEDLKVIADAIIAVLKKRFPTAFNSAAQAAVTKALQQFIDALEKEFKKL',
  'sequence_length': 149,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb04',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb04',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'GLTAEEQALVRAIWAKVREDLEGFGLAVLLKTFTEHPETLKKFPRFKDLKSEEEILASEKAKKHGVTVLTALFAIFDKGENFEEEIKPLAESHYKEHKIPISDLKVIADAIVAVLKEAFPEAFDAKAQAAFTKALEQFIKAFEEEYKKL',
  'sequence_length': 149,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb05',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb05',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'GLSDEEQALVLSIFEKVKEDLAGFGLDVLLLAFTKNPATLEKFPRFADLKSEAELLASEKAKEHGITVLTALFAIFEKGDDFDAEVEPLATSHTREHKIPTSDLEVIAAAILETAKERFPTEFDEEAQAALEKALAQFIAAYAAQAAKL',
  'sequence_length': 149,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb06',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb06',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'KLSEEEKEIVLKIFELVEKDVEEIGLRVLELTFEKHPETLEKFPRLRELLAAGRLEELEAYLREHGVTVLKALIEAIKNEDEELLEKLAKSHKEEHKIPIEYLKYIADSIIEVLEEKFPKEFNEKAREALKKALEYFIEELEKYYK',
  'sequence_length': 146,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb07',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb07',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'KLTPEEKAIVLRIFALVREDRAGIGAAILRRTFEAHPETLEKFPRLRALRAAGREAELEALLREHGVTVLDALIEIVENDDEELLKKLAESHKTTHKIPIEHLEHIAAALLEVLAEKYPEEFGPEARAAVTKALELFIKKLAEFYE',
  'sequence_length': 146,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb08',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb08',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'IKISEEEFEIVLEIFELVKKDLAGIGKEVLILTFTKHPETLKKFPRFAHLKTVEELEASPLLAEHGVTVLKALIKIVEELKKGDTSLIKELAKSHKTEHKIDIKDLKYIAESIIEVLKKRFPEEFDEKAKEAVEKVLNLFIEKIEEFYKK',
  'sequence_length': 150,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb09',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb09',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'SKLTEEEWKTVFKIFALVEKDLEGFGLAVLIRTFTRYPETLKKFPRFAHLKTVEELRASPLLREHGVTVLKALTKIAEELKKGKTGTLKKLAESHSKVHKIPISDLERIAEAIIEVLEERFPEEFDEKAKEAVKKFLDLFIEKHAEFVKK',
  'sequence_length': 150,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb10',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb10',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'DAEKQALVASIFAKFEADLEGFGKAVLIKTFTKHPETRKKFPRFKHLKSVEELEKSEELKEHGVTVLTALREISLGENQDKKIKDLATSHKEKHKIPIEDLEVIAAAILEVAKERFPEEFDEAAQAALQEFLDDFISKLKEYFE',
  'sequence_length': 144,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb11',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb11',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'KLSEEEKEIVLKIFALVEKDLEGFGKEVLIKTFLKYPETLKKFPRFKHLKTEEELKASEELKEHGVTVLKALIEIFKNEDEEKLKELAKSHKEEHKIPIEDLEKIAEAIIEVLKEKYPEEFDEEAEEAVKKFLKLFIEKLKEYRE',
  'sequence_length': 145,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb12',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb12',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'LNLSPEDKAKVLEIFALVEEDLEGFGREVLILTFTKHPETLKKFPRFAHLKTEEELRASEELKEHGVTVLKALRAILEKGDEELLKKLAESHTKEHKIPVSDLEVIAESIIEVAKKRFPEEFGEEAQAALKKFLEEFIEKWKEYQEEFK',
  'sequence_length': 149,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb13',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb13',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'DERNKLVLSAFALVREDLEEIGAEVLILTFTENPETLKKFPRFAHLKTEEELKKSPLLKEHGVTVLNALIEIAELKYSGGDYESLVKELAKSHKEKHKIPIEDLKAIAEAILKVLKKRYPEEFGEKTQAALKEFLDEFIELTEKYYK',
  'sequence_length': 147,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb14',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb14',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'EKEKNELVLKAFELIEKDLEGFGSEVLILTFTKHPETLKKFPRFKHLKTEEEFKASEELKEHGVTVLKALIEIAKLKVSGEDYDSLIKELAKSHKTKHKIPIEYLKYIADAILEVAKKRFPKEFDEKTYAALKEFLDYFIEKIEKYYK',
  'sequence_length': 148,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb15',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb15',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'EEEEKQIVLELFAKVEEDLEGIGLEVLILTFTKHPETRKKFPRFAHLTTEAQLQASPELKQHGVTVLKALITIAKLYYEGKDYESLIKELAKSHKEEHKIPIEYLEYISESILEVLKKRFPEFFGEKAQAAVRKFLDFFISKLKEYYE',
  'sequence_length': 148,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb16',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb16',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'AEEEKEKVLSIFKLVEKDKKTIGSEVLIITFTKNPETKKKFPRFKDLKTVEELKASEKVKDHGVTVLDALIEWARLHVEGKDYDSLVKKLAESHKKEHKIPIEDLKSIADALIEVLKKFYPEEFGEEAQAAVQKLLNYFIEKLKQYYE',
  'sequence_length': 148,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb17',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb17',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'IKLSEEEKKLVLEIFKLFEENLEEFGKEVLITTFTKHPETKKKFPRFAHLKTEEEFLASPELAKHGVTVLNALIEIAKLYLEGKDYRSLIKKLAKSHKLEHKIPIEDLKYIADAIIEVAKKFFPEKFGEKAQEALKKFLNYFIEELEKEYEKL',
  'sequence_length': 153,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb18',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb18',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'DEEKKKLVLEAFELVEKDIEGIGAEVLKLTFEKHPETLEKFPRLKELHAAGSPELEELLKEHGATVLKALIEIARLKISGGDYLSLVKELAKSHKEEHKIPIEDLKKIAEALLEVLKEKYPEEFGEETQEALKEFLDWFIEELEKEFKE',
  'sequence_length': 149,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb19',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb19',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'SEEKAALVLALFDRVEADREEIGAAVLRRTFEEHPETLKKFPRFLELYKKGSPELDALLKEHGKTVLDALIEIARLRYSGEDYRSLIKELAKSHKEEHKIPIEDLRHIAEALLAVLAERFPDEFGPEARAALTDFLDWFIAEIEEEYKK',
  'sequence_length': 149,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb20',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb20',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'LSEEEWKIVLEIFALVREDLAGVGAAVLERTFATHPETLKKFPRFLAAAEAGVLDRALLAAHGETVLTALIEIAESKLDPELIKKLAESHVKEHKIPIEYLRAIADSLIAVLKERYPERFGEKAQEAVKKFLDLFIEKFEEEAEKEK',
  'sequence_length': 147,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'dnMb21',
  'protein_name': 'ProteinMPNN-designed myoglobin dnMb21',
  'uniprot_accession': '',
  'dataset_group': 'AI_DESIGNED_POSITIVE',
  'reference_family_or_control_type': 'Designed myoglobin',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10811672/',
  'amino_acid_sequence': 'SLTPEELAIVKALFARVREDLEGVGAEVLRLTFEKHPETLKKFPRFLELKKAGSPELEAELRAHGVTVLTALIELADNYEGNNETLEKLAESHTKVHKIPVSDLKNIAAAIIEVLKERFPEEFGEEAQAAFTKFLDKFIKDIAELQKKFE',
  'sequence_length': 150,
  'sequence_source': 'Supplementary PDF pp 26-27; cross-checked with ZIP PDB models',
  'sequence_notes': 'Core designed protein sequence; expression tags and His-tag excluded. Heme-binding supported by '
                    'published spectroscopy; not oxygen-binding proof.',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P9WKH3',
  'protein_name': 'MhuD',
  'uniprot_accession': 'P9WKH3',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Heme oxygenase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P9WKH3',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P23847',
  'protein_name': 'DppA',
  'uniprot_accession': 'P23847',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Dipeptide-binding protein',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P23847',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P0ABI8',
  'protein_name': 'CyoB',
  'uniprot_accession': 'P0ABI8',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Cytochrome bo3 oxidase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P0ABI8',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q9KL41',
  'protein_name': 'HutZ',
  'uniprot_accession': 'Q9KL41',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Heme oxygenase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q9KL41',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q2FZE9',
  'protein_name': 'IsdA',
  'uniprot_accession': 'Q2FZE9',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Heme acquisition protein',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q2FZE9',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q9I609',
  'protein_name': 'NirN',
  'uniprot_accession': 'Q9I609',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Heme d1 metabolism',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q9I609',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'I6X8R5',
  'protein_name': 'Rv0203',
  'uniprot_accession': 'I6X8R5',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Heme-binding protein',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/I6X8R5',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P32339',
  'protein_name': 'HMX1',
  'uniprot_accession': 'P32339',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Heme oxygenase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P32339',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P12946',
  'protein_name': 'CtaA',
  'uniprot_accession': 'P12946',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Heme A synthase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P12946',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P09601',
  'protein_name': 'HMOX1',
  'uniprot_accession': 'P09601',
  'dataset_group': 'NATURAL_POSITIVE',
  'reference_family_or_control_type': 'Heme oxygenase',
  'heme_binding_label': 'positive',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P09601',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P00698',
  'protein_name': 'Hen egg-white lysozyme',
  'uniprot_accession': 'P00698',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Standard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P00698',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P63279',
  'protein_name': 'SUMO-conjugating enzyme UBC9',
  'uniprot_accession': 'P63279',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Standard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P63279',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P9WMH1',
  'protein_name': 'IdeR',
  'uniprot_accession': 'P9WMH1',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Context-specific assay negative',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P9WMH1',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P08246',
  'protein_name': 'Neutrophil elastase',
  'uniprot_accession': 'P08246',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Standard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P08246',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q86V81',
  'protein_name': 'ALYREF',
  'uniprot_accession': 'Q86V81',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Standard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q86V81',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q99836',
  'protein_name': 'MyD88',
  'uniprot_accession': 'Q99836',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Standard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q99836',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P36873',
  'protein_name': 'PP1-gamma',
  'uniprot_accession': 'P36873',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Standard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P36873',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P25685',
  'protein_name': 'DNAJB1',
  'uniprot_accession': 'P25685',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Standard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P25685',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q6FHJ7',
  'protein_name': 'SFRP4',
  'uniprot_accession': 'Q6FHJ7',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Standard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q6FHJ7',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'Q02750',
  'protein_name': 'MEK1',
  'uniprot_accession': 'Q02750',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Standard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/Q02750',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P02246',
  'protein_name': 'Hemerythrin (Themiste dyscritum)',
  'uniprot_accession': 'P02246',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P02246',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P72509',
  'protein_name': 'C-phycocyanin alpha (Arthrospira)',
  'uniprot_accession': 'P72509',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P72509',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P00282',
  'protein_name': 'Azurin',
  'uniprot_accession': 'P00282',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P00282',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P61949',
  'protein_name': 'Flavodoxin',
  'uniprot_accession': 'P61949',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P61949',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P0ABQ4',
  'protein_name': 'Dihydrofolate reductase',
  'uniprot_accession': 'P0ABQ4',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P0ABQ4',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P00918',
  'protein_name': 'Carbonic anhydrase II',
  'uniprot_accession': 'P00918',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P00918',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P42212',
  'protein_name': 'Green fluorescent protein',
  'uniprot_accession': 'P42212',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P42212',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P0AA25',
  'protein_name': 'Thioredoxin',
  'uniprot_accession': 'P0AA25',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P0AA25',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P61823',
  'protein_name': 'Ribonuclease A',
  'uniprot_accession': 'P61823',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P61823',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''},
 {'record_id': 'P22629',
  'protein_name': 'Streptavidin',
  'uniprot_accession': 'P22629',
  'dataset_group': 'NEGATIVE_CONTROL',
  'reference_family_or_control_type': 'Hard control',
  'heme_binding_label': 'negative',
  'evidence_source': 'https://www.uniprot.org/uniprotkb/P22629',
  'amino_acid_sequence': '',
  'sequence_length': '',
  'sequence_source': 'UniProt: to download in Colab',
  'sequence_notes': '',
  'split': 'HOLDOUT_UNASSIGNED',
  'uniprot_fasta_header': ''}]

# Count the records in each group.
counts = Counter(record["dataset_group"] for record in records)
print("Protein records:", len(records))
print("Breakdown:", dict(counts))
print("Sequences already embedded:", sum(bool(r["amino_acid_sequence"]) for r in records))
assert len(records) == 80
assert counts == {"REFERENCE": 30, "AI_DESIGNED_POSITIVE": 20, "NATURAL_POSITIVE": 10, "NEGATIVE_CONTROL": 20}


In [ ]:
# Step 2: Download the 60 non-designed sequences from UniProt.
# Note: Google Colab needs an internet connection for this step.
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Retry when UniProt is temporarily busy, rather than failing immediately.
session = requests.Session()
retry_rules = Retry(total=4, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry_rules))

# Collect errors instead of silently treating a missing sequence as valid.
errors = []
for position, record in enumerate(records, start=1):
    # Designed protein sequences are embedded already, so skip downloading them.
    if record["amino_acid_sequence"]:
        continue

    # Construct the FASTA download link from its exact UniProt accession.
    accession = record["uniprot_accession"]
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        # Download the source record; never guess a sequence from its name.
        response = session.get(url, timeout=35)
        response.raise_for_status()
        lines = [line.strip() for line in response.text.splitlines() if line.strip()]

        # Validate the FASTA format and make sure the returned accession matches.
        if len(lines) < 2 or not lines[0].startswith(">"):
            raise ValueError("UniProt did not return a valid FASTA record")
        fasta_accession = lines[0].split("|")[1]
        if fasta_accession != accession:
            raise ValueError(f"Accession mismatch: expected {accession}, got {fasta_accession}")

        # Combine the sequence lines into one amino-acid string.
        sequence = "".join(lines[1:]).upper()
        valid_letters = set("ACDEFGHIKLMNPQRSTVWYBXZUO")
        if not sequence or not set(sequence).issubset(valid_letters):
            raise ValueError("Unexpected amino-acid characters in sequence")

        # Save the verified accession, sequence length and original FASTA header.
        record["amino_acid_sequence"] = sequence
        record["sequence_length"] = len(sequence)
        record["uniprot_fasta_header"] = lines[0]
        record["sequence_source"] = url
        print(f"{position:02d}/80  {accession}  {len(sequence)} aa  OK")

    except Exception as exc:
        errors.append((accession, str(exc)))
        print(f"{position:02d}/80  {accession}  ERROR: {exc}")

    # Avoid flooding UniProt with requests.
    time.sleep(0.15)

# Don't claim a complete dataset when one or more downloads failed.
if errors:
    raise RuntimeError(f"{len(errors)} downloads failed: {errors}. Rerun this cell after fixing errors.")

print("All 60 UniProt sequences downloaded successfully.")


In [ ]:
# Step 3: Validate and export the complete 80-protein dataset.

# Check every protein now has a non-empty amino-acid sequence.
missing = [r["record_id"] for r in records if not r["amino_acid_sequence"]]
if missing:
    raise ValueError(f"Missing sequences, cannot export complete dataset: {missing}")

# Check sequence lengths agree with the actual strings.
for record in records:
    assert len(record["amino_acid_sequence"]) == int(record["sequence_length"])

# Make sure all record IDs are unique.
assert len({r["record_id"] for r in records}) == 80

# Make a table with all metadata and sequences.
df = pd.DataFrame(records)

# Save the primary master dataset CSV.
df.to_csv("master_dataset_80_proteins.csv", index=False)

# Save the sequences as FASTA for later protein analysis.
with open("master_dataset_80_proteins.fasta", "w") as output:
    for r in records:
        header = f">{r['record_id']}|{r['dataset_group']}|{r['protein_name']}"
        output.write(header.replace(" ", "_") + "\n")
        sequence = r["amino_acid_sequence"]
        for start in range(0, len(sequence), 60):
            output.write(sequence[start:start + 60] + "\n")

# Show a compact summary without filling the screen with long sequences.
print("TOTAL:", len(df), "proteins")
print("GROUPS:\n", df.groupby("dataset_group").size())
display(df[["record_id", "protein_name", "dataset_group", "heme_binding_label", "sequence_length"]].head(12))
print("Saved: master_dataset_80_proteins.csv")
print("Saved: master_dataset_80_proteins.fasta")


In [ ]:
# Step 4: Download both finished files to your computer.
# Only run this after Step 3 has completed successfully.
from google.colab import files
files.download("master_dataset_80_proteins.csv")
files.download("master_dataset_80_proteins.fasta")
